# SOXX 2018-09 原始普通股集中度

按冻结的来源、原件 SHA、表格字节范围解析普通股数量和美元市值，核对印刷合计。0 收益读取、0 拟合、0 账户。

In [1]:
import json,hashlib,re
from pathlib import Path
from datetime import datetime,timezone
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_2018_sep_concentration_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(root/plan['parent_body_identity'])==plan['parent_body_identity_sha256'] and sha(root/plan['parent_body_transport'])==plan['parent_body_transport_sha256']
assert json.loads((root/plan['parent_body_transport']).read_text())['nas_archive_sha256']==plan['source_NAS_archive_sha256']
assert not (card/'attempt_started.json').exists() and not (card/'result.json').exists()
(card/'attempt_started.json').write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'original_body_reads_started':1},indent=2)+'\n')
payload=(root/plan['raw_path']).read_bytes();assert len(payload)==plan['raw_bytes'] and hashlib.sha256(payload).hexdigest()==plan['raw_sha256']
a,b=plan['bounded_section_byte_range'];section=payload[a:b];assert hashlib.sha256(section).hexdigest()==plan['bounded_section_sha256']
integer=re.compile(r'^[0-9][0-9,]*$');rows=BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser').find_all('tr');parsed=[];ambiguous=[];started=False;total_label=False;printed=None
for row in rows:
 cells=[x.get_text(' ',strip=True).replace('\xa0',' ').strip() for x in row.find_all(['td','th'],recursive=False)];nonempty=[x for x in cells if x and x!='$']
 if not nonempty:continue
 name=nonempty[0];upper=name.upper();nums=[int(x.replace(',','')) for x in nonempty[1:] if integer.fullmatch(x)]
 if not started:
  if 'COMMON STOCKS' in upper and 'TOTAL' not in upper:started=True
  continue
 if 'TOTAL COMMON STOCKS' in upper:
  total_label=True
  if nums:printed=nums[-1];break
  continue
 if total_label:
  if nums:printed=nums[-1]
  break
 if '—' in name or '–' in name or 'COMMON STOCKS' in upper:continue
 if len(nums)>=2 and any(ch.isalpha() for ch in name):parsed.append({'security_original':name,'shares_original':nums[0],'fair_value_usd_original':nums[-1]})
 elif any(ch.isalpha() for ch in name) and nums:ambiguous.append({'row_text':nonempty[:5],'numeric_tokens':nums})
values=[x['fair_value_usd_original'] for x in parsed];total=sum(values)
for x in parsed:x['reported_equity_weight']=x['fair_value_usd_original']/total if total else None
checks={'common_stocks_heading_found':started,'total_label_found':total_label,'printed_total_found':printed is not None,'row_count_20_to_40':20<=len(parsed)<=40,'all_positive':all(x['shares_original']>0 and x['fair_value_usd_original']>0 for x in parsed),'distinct_names':len({x['security_original'] for x in parsed})==len(parsed),'total_closure_within_1_usd':printed is not None and abs(total-printed)<=1,'ambiguous_rows_zero':len(ambiguous)==0}
ranked=sorted(parsed,key=lambda x:x['fair_value_usd_original'],reverse=True)
result={'status':'2018_Sep_original_SOXX_reported_equity_values_closed' if all(checks.values()) else '2018_Sep_original_SOXX_numeric_source_unqualified','plan_sha256':sha(card/'plan.json'),'report_period':plan['report_period'],'accession':plan['accession'],'filed':plan['filed'],'accepted_at_utc':plan['accepted_at_utc'],'raw_sha256':plan['raw_sha256'],'section_sha256':plan['bounded_section_sha256'],'parsed_equity_rows':len(parsed),'equities':parsed,'ambiguous_rows':ambiguous,'parsed_fair_value_total_usd':total,'printed_total_common_stocks_usd':printed,'total_closure_difference_usd':total-printed if printed is not None else None,'checks':checks,'top5_disclosed_equity_value_share':sum(x['reported_equity_weight'] for x in ranked[:5]) if total else None,'herfindahl_sum_squared_disclosed_equity_weights':sum(x['reported_equity_weight']**2 for x in parsed) if total else None,'source_NAS_archive_sha256':plan['source_NAS_archive_sha256'],'source_NAS_scoped_restore_verified':True,'HTTP_GETs_new':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'scientific_boundary':'Source value only; no official daily index weights, Qlib calendar coverage, incremental information or funded Stage1 account.'}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','parsed_equity_rows','total_closure_difference_usd','top5_disclosed_equity_value_share','herfindahl_sum_squared_disclosed_equity_weights')})


{'status': '2018_Sep_original_SOXX_reported_equity_values_closed', 'parsed_equity_rows': 30, 'total_closure_difference_usd': 0, 'top5_disclosed_equity_value_share': 0.4193657783329641, 'herfindahl_sum_squared_disclosed_equity_weights': 0.05320961520954554}


## 中文结论

2018-09-30 SOXX 原件的 30 条普通股原始美元市值合计 1,637,362,509，与印刷 Total Common Stocks 差 0。披露股内前五占比 0.419366，HHI 0.053210。源原件经 NAS 回读，0 市场收益读取、0 拟合、0 账户。仅为 2019 年初 Qlib 来源前缀提供候选滞后披露值；尚未映射，也不证明 SOXX 超额。